# Lezione 32 — Il blocco Transformer

> **Modulo:** Transformer e modello open · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezione 31 (self-attention).
>
> Obiettivo pratico unico: assemblare un **blocco Transformer encoder**
> completo in NumPy — self-attention + connessione residua + layer
> normalization + rete feed-forward — e verificare che la forma della
> sequenza si conservi.

## Parte 1 — I quattro ingredienti

La self-attention (Lezione 31) e' il cuore, ma da sola non basta. Il blocco
Transformer encoder la avvolge in tre altri ingredienti (Vaswani et al., 2017,
Sez. 3.1 e 5.4):

1. **Connessione residua** — si somma l'input all'output del sotto-strato
   (`x + Sublayer(x)`): aiuta il gradiente a fluire e permette allo strato di
   imparare una *modifica* invece di ricostruire tutto da capo.
2. **Layer normalization** — normalizza ogni vettore (media 0, varianza 1 sulle
   sue feature) per stabilizzare l'addestramento.
3. **Rete feed-forward posizionale (FFN)** — due trasformazioni lineari con una
   non-linearita' in mezzo, applicate **indipendentemente a ogni posizione**.

Struttura di un blocco (variante *post-norm* dell'articolo originale):

```text
x -> [self-attention] -> +x -> LayerNorm -> [FFN] -> +. -> LayerNorm -> out
```

La forma entra `(n, d_model)` ed **esce identica** `(n, d_model)`: per questo i
blocchi si possono impilare.

In [1]:
import numpy as np

rng = np.random.default_rng(32)


def softmax(s, axis=-1):
    s = s - np.max(s, axis=axis, keepdims=True)
    e = np.exp(s)
    return e / np.sum(e, axis=axis, keepdims=True)


def layer_norm(x, eps=1e-5):
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps)


def self_attention(X, Wq, Wk, Wv):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    A = softmax(Q @ K.T / np.sqrt(K.shape[-1]), axis=-1)
    return A @ V, A


def relu(x):
    return np.maximum(0.0, x)

In [2]:
def transformer_block(X, params):
    # 1. self-attention + residua + norm
    attn, A = self_attention(X, params["Wq"], params["Wk"], params["Wv"])
    attn = attn @ params["Wo"]              # proietta di nuovo a d_model
    x1 = layer_norm(X + attn)               # residua + layer norm
    # 2. feed-forward posizionale + residua + norm
    ff = relu(x1 @ params["W1"] + params["b1"]) @ params["W2"] + params["b2"]
    out = layer_norm(x1 + ff)
    return out, A


def init_params(d_model, d_k, d_ff, seed=32):
    r = np.random.default_rng(seed)
    s = 0.3
    return {
        "Wq": r.normal(size=(d_model, d_k)) * s,
        "Wk": r.normal(size=(d_model, d_k)) * s,
        "Wv": r.normal(size=(d_model, d_k)) * s,
        "Wo": r.normal(size=(d_k, d_model)) * s,
        "W1": r.normal(size=(d_model, d_ff)) * s,
        "b1": np.zeros(d_ff),
        "W2": r.normal(size=(d_ff, d_model)) * s,
        "b2": np.zeros(d_model),
    }

## Parte 2 — Una memoria attraverso il blocco

In [3]:
frase = "The user prefers morning sessions"
tokens = frase.lower().split()
d_model = 16


def embed_token(tok, dim=d_model):
    v = np.zeros(dim)
    for i, ch in enumerate(tok):
        v[(ord(ch) + i) % dim] += 1.0
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else v


X = np.vstack([embed_token(t) for t in tokens])
params = init_params(d_model=d_model, d_k=16, d_ff=32)

out, A = transformer_block(X, params)
print("input  :", X.shape)
print("output :", out.shape, "-> stessa forma, i blocchi si impilano")
print("layer norm sull'output? media~0, std~1 per riga:")
print("  media per token:", np.round(out.mean(axis=1), 3))
print("  std   per token:", np.round(out.std(axis=1), 3))

input  : (5, 16)
output : (5, 16) -> stessa forma, i blocchi si impilano
layer norm sull'output? media~0, std~1 per riga:
  media per token: [-0.  0.  0. -0.  0.]
  std   per token: [1. 1. 1. 1. 1.]


Leggi l'output: la forma entra e esce `(5, 16)`. Dopo la layer norm finale
ogni riga (token) ha media ~0 e deviazione standard ~1 — esattamente cio' che
la normalizzazione garantisce. Ecco perche' si possono **impilare** decine di
blocchi identici.

In [4]:
# impiliamo 3 blocchi: l'output di uno e' l'input del successivo
h = X
for strato in range(3):
    h, A = transformer_block(h, params)
    print(f"dopo blocco {strato + 1}: forma {h.shape}, "
          f"norma media {np.linalg.norm(h, axis=1).mean():.3f}")

dopo blocco 1: forma (5, 16), norma media 4.000
dopo blocco 2: forma (5, 16), norma media 4.000
dopo blocco 3: forma (5, 16), norma media 4.000


## Parte 3 — Il progetto: Memory AI Lab, passo 32

Impacchetto un **encoder** (pila di blocchi) che trasforma una memoria in una
sequenza di vettori contestualizzati e in un unico vettore-riassunto (media dei
token) — l'analogo "Transformer" del sentence-embedding della Lezione 17.

In [5]:
def encoder_memoria(frase, n_blocchi=2, d_model=16, seed=32):
    tokens = frase.lower().split()
    X = np.vstack([embed_token(t, d_model) for t in tokens])
    params = init_params(d_model=d_model, d_k=16, d_ff=32, seed=seed)
    h = X
    for _ in range(n_blocchi):
        h, _ = transformer_block(h, params)
    riassunto = h.mean(axis=0)          # pooling: un vettore per la frase
    return h, riassunto


seq, riassunto = encoder_memoria("Marco visited Glasgow with his son")

# controllo di non-regressione: forma conservata e riassunto della dim attesa
assert seq.shape[1] == 16
assert riassunto.shape == (16,)
print("sequenza contestualizzata:", seq.shape)
print("vettore-riassunto della memoria:", riassunto.shape,
      "| norma:", round(float(np.linalg.norm(riassunto)), 4))

sequenza contestualizzata: (6, 16)
vettore-riassunto della memoria: (16,) | norma: 3.5622


## Riepilogo (max 8 punti)

1. Il blocco Transformer = self-attention + residua + layer norm + FFN.
2. **Residua** (`x + Sublayer(x)`): il gradiente fluisce, lo strato impara una
   correzione.
3. **Layer norm**: ogni vettore a media 0 / varianza 1, addestramento stabile.
4. **FFN posizionale**: due lineari + non-linearita', per ogni posizione
   indipendentemente.
5. La forma entra ed esce identica `(n, d_model)`.
6. Per questo i blocchi si **impilano** (encoder profondo).
7. Un pooling finale da' un vettore-riassunto della memoria.
8. E' l'architettura che modelli come Gemma ripetono molte volte.

## Quiz

1. Perche' la connessione residua richiede che `Sublayer(x)` abbia la stessa
   forma di `x`?
2. Cosa normalizza la layer norm: le feature di un token o i token tra loro?
3. Perche' l'FFN e' detta "posizionale"?

*(Risposte: 1. perche' si sommano elemento per elemento; 2. le feature di ogni
singolo token, indipendentemente dagli altri; 3. perche' applica gli stessi
pesi a ogni posizione separatamente, senza mescolare i token.)*